# Генерация и полный замер на одной T4 в Kaggle
Выберите GPU T4 x2 и Internet, затем Run All. Используется только GPU 0.
Веса из релиза v1.0 и 1919 вопросов из зафиксированного коммита GitHub
скачиваются автоматически до начала замера; проверяются SHA256.
Можно также прикрепить Dataset с checkpoint_130m.zip и questions_used.jsonl.


In [ ]:
%pip install -q "transformers==4.48.3" "tokenizers==0.21.0" "huggingface-hub==0.36.2" "safetensors>=0.4.5"


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import urllib.request
import zipfile

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
assert INPUT.is_dir() and WORK.is_dir(), 'Запустите ноутбук в Kaggle'
os.chdir(WORK)
hardware = subprocess.check_output([
    'nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
    '--format=csv,noheader'], text=True)
print(hardware)
assert hardware.splitlines() and hardware.splitlines()[0].split(',')[1].strip() in {
    'Tesla T4', 'NVIDIA Tesla T4', 'NVIDIA T4', 'T4'}, 'Нужна GPU T4'
Path('hardware_preflight.txt').write_text(hardware, encoding='utf-8')

def find_unique(name):
    matches = sorted(INPUT.rglob(name))
    assert len(matches) <= 1, f'Найдено несколько {name}; оставьте один Dataset'
    return matches[0] if matches else None

def sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

def download(url, destination, expected_sha256):
    if destination.exists() and sha256(destination) == expected_sha256:
        return
    temporary = destination.with_suffix('.download')
    print('Скачивание', destination.name, flush=True)
    with urllib.request.urlopen(url, timeout=120) as response, temporary.open('wb') as stream:
        shutil.copyfileobj(response, stream, length=4 * 1024 * 1024)
    assert sha256(temporary) == expected_sha256, f'Неверный SHA256: {destination.name}'
    temporary.replace(destination)

questions = find_unique('questions_used.jsonl')
if questions:
    shutil.copy2(questions, WORK / questions.name)
else:
    download('https://raw.githubusercontent.com/Kera-0/hw-2-/043d2689b658687e8dce7c8c53b83c99311b79b2/questions_used.jsonl',
             WORK / 'questions_used.jsonl',
             'a3a128e5acf694f9b9a828dcc582bacb79367eecf11af2f6041c0c5f788e7003')
archive = find_unique('checkpoint_130m.zip')
if archive:
    shutil.copy2(archive, WORK / archive.name)
else:
    weights = find_unique('model.safetensors')
    if weights:
        checkpoint = weights.parent
        names = ['config.json', 'generation_config.json', 'model.safetensors',
                 'tokenizer.json', 'training_meta.json', 'inference_config.json']
        assert all((checkpoint / name).is_file() for name in names), 'Неполный checkpoint'
        with zipfile.ZipFile(WORK / 'checkpoint_130m.zip', 'w', compression=zipfile.ZIP_STORED) as target:
            for name in names:
                target.write(checkpoint / name, 'best/' + name)
    else:
        download('https://github.com/Kera-0/hw-2-/releases/download/v1.0/checkpoint_130m.zip',
                 WORK / 'checkpoint_130m.zip',
                 'adf2134a8c637a7b50b0fbee3370475e2e64c719eaf29fe29f73104f8582abf7')
print('Входные файлы готовы:', WORK)


In [ ]:
from pathlib import Path
import os
BASE = Path('/kaggle/working')
os.chdir(BASE)
QUESTIONS_FILE = 'questions_used.jsonl'
for name in ['checkpoint_130m.zip', QUESTIONS_FILE]:
    assert Path(name).is_file(), f'Отсутствует {name}'
INFERENCE_PROGRAM = '"""Standalone cold-load inference benchmark, strict NVIDIA T4 by default.\n\nDependency installation and obtaining input artifacts precede the timed run.\nThe measurement includes imports, checkpoint extraction/loading, tokenization,\nGPU generation, and writing/checking the 1919-line output. No datasets library.\n"""\nimport argparse\nimport json\nfrom pathlib import Path\nimport time\nimport zipfile\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--checkpoint\', default=\'checkpoint_130m.zip\')\n    parser.add_argument(\'--questions\', default=\'questions_used.jsonl\')\n    parser.add_argument(\'--output\', default=\'submission_130m.jsonl\')\n    parser.add_argument(\'--allow-other-gpu\', action=\'store_true\')\n    parser.add_argument(\'--batch-size\', type=int)\n    parser.add_argument(\'--max-new-tokens\', type=int)\n    args = parser.parse_args()\n    begin = time.perf_counter()\n    import os\n    os.environ.setdefault(\'CUDA_VISIBLE_DEVICES\', \'0\')\n    os.environ.setdefault(\'USE_TF\', \'0\')\n    os.environ.setdefault(\'USE_FLAX\', \'0\')\n    os.environ.setdefault(\'TOKENIZERS_PARALLELISM\', \'false\')\n    import math\n    import hashlib\n    import ast\n    import platform\n    import torch\n    from tokenizers import Tokenizer\n    from transformers import BartForConditionalGeneration\n    import transformers\n    phases = {\'imports\': time.perf_counter() - begin}\n    torch.set_num_threads(4)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'Select a GPU runtime\')\n    gpu = torch.cuda.get_device_name()\n    capability = torch.cuda.get_device_capability()\n    actual_t4 = gpu in {\'Tesla T4\', \'NVIDIA Tesla T4\', \'NVIDIA T4\', \'T4\'} and capability == (7, 5)\n    if not args.allow_other_gpu and not actual_t4:\n        raise RuntimeError(f\'This submission benchmark requires an actual T4; found {gpu}\')\n    torch.cuda.reset_peak_memory_stats()\n    phase_started = time.perf_counter()\n    archive_path = Path(args.checkpoint)\n    checkpoint_sha256 = None\n    if archive_path.is_file():\n        digest = hashlib.sha256()\n        with archive_path.open(\'rb\') as stream:\n            for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                digest.update(chunk)\n        checkpoint_sha256 = digest.hexdigest()\n        target = Path(\'checkpoint_130m_extracted\')\n        with zipfile.ZipFile(archive_path) as archive:\n            allowed = {\'best/\' + n for n in [\'config.json\', \'generation_config.json\', \'model.safetensors\',\n                       \'tokenizer.json\', \'training_meta.json\', \'inference_config.json\']}\n            files = [item.filename for item in archive.infolist() if not item.is_dir()]\n            directories = {item.filename for item in archive.infolist() if item.is_dir()}\n            if set(files) != allowed or len(files) != len(allowed) or not directories <= {\'best/\'}:\n                raise ValueError(\'Unexpected checkpoint archive files\')\n            for name in sorted(allowed):\n                archive.extract(name, target)\n        checkpoint = target / \'best\'\n    else:\n        checkpoint = archive_path\n    phases[\'checkpoint_extraction_and_hash\'] = time.perf_counter() - phase_started\n    phase_started = time.perf_counter()\n    settings = json.loads((checkpoint / \'inference_config.json\').read_text(encoding=\'utf-8\'))\n    if args.batch_size is not None:\n        settings[\'batch_size\'] = args.batch_size\n    if args.max_new_tokens is not None:\n        settings[\'max_new_tokens\'] = args.max_new_tokens\n    if settings[\'batch_size\'] < 1 or not 1 <= settings[\'max_new_tokens\'] <= settings[\'max_target\']:\n        raise ValueError(\'Invalid batch size or generation length\')\n    questions_bytes = Path(args.questions).read_bytes()\n    normalized_questions_sha256 = hashlib.sha256(questions_bytes.replace(b\'\\r\\n\', b\'\\n\')).hexdigest()\n    if normalized_questions_sha256 != \'a3a128e5acf694f9b9a828dcc582bacb79367eecf11af2f6041c0c5f788e7003\':\n        raise ValueError(\'Questions or their order differ from the repository test input\')\n    rows = [json.loads(line) for line in questions_bytes.decode(\'utf-8\').splitlines()]\n    questions = [row[\'question\'] for row in rows]\n    if len(questions) != 1919 or any(not isinstance(q, str) or not q.strip() for q in questions):\n        raise ValueError(\'Expected 1919 nonempty questions in original order\')\n    tokenizer = Tokenizer.from_file(str(checkpoint / \'tokenizer.json\'))\n    model = BartForConditionalGeneration.from_pretrained(\n        str(checkpoint), local_files_only=True, torch_dtype=torch.float16,\n        attn_implementation=\'sdpa\').cuda().eval()\n    parameters = sum(p.numel() for p in model.parameters())\n    if parameters != 130_183_936:\n        raise ValueError(\'Wrong model checkpoint\')\n    if (model.config.pad_token_id, model.config.eos_token_id) != (0, 3):\n        raise ValueError(\'Checkpoint special tokens do not match this inference program\')\n    torch.cuda.synchronize()\n    phases[\'input_and_model_load\'] = time.perf_counter() - phase_started\n    phase_started = time.perf_counter()\n    encoded = tokenizer.encode_batch(questions, add_special_tokens=False)\n    source_clipped = sum(len(x.ids) + 1 > settings[\'max_source\'] for x in encoded)\n    sequences = [x.ids[:settings[\'max_source\'] - 1] + [3] for x in encoded]\n    order = sorted(range(len(sequences)), key=lambda i: len(sequences[i]))\n    predictions = [None] * len(questions)\n    no_eos = 0\n    batch_size = settings[\'batch_size\']\n    phases[\'tokenization\'] = time.perf_counter() - phase_started\n    phase_started = time.perf_counter()\n    with torch.inference_mode():\n        for start in range(0, len(order), batch_size):\n            indices = order[start:start + batch_size]\n            width = math.ceil(max(len(sequences[i]) for i in indices) / 8) * 8\n            ids = torch.zeros((len(indices), width), dtype=torch.long)\n            for j, index in enumerate(indices):\n                ids[j, :len(sequences[index])] = torch.tensor(sequences[index], dtype=torch.long)\n            ids = ids.to(\'cuda\')\n            generated = model.generate(input_ids=ids, attention_mask=ids.ne(0).long(),\n                                       do_sample=False, num_beams=1, use_cache=True,\n                                       max_new_tokens=settings[\'max_new_tokens\'], forced_eos_token_id=None,\n                                       suppress_tokens=[0, 1, 2]).cpu().tolist()\n            for index, tokens in zip(indices, generated):\n                predictions[index] = tokenizer.decode(tokens, skip_special_tokens=True)\n                no_eos += 3 not in tokens[1:]\n    torch.cuda.synchronize()\n    phases[\'generation_and_decode\'] = time.perf_counter() - phase_started\n    phase_started = time.perf_counter()\n    output = Path(args.output)\n    output.write_text(\'\'.join(json.dumps({\'code\': code}, ensure_ascii=False) + \'\\n\' for code in predictions), encoding=\'utf-8\', newline=\'\\n\')\n    checked = [json.loads(line) for line in output.read_text(encoding=\'utf-8\').splitlines()]\n    assert len(checked) == 1919 and all(set(row) == {\'code\'} and isinstance(row[\'code\'], str) for row in checked)\n    syntax_error_rows = []\n    for index, row in enumerate(checked, 1):\n        try:\n            ast.parse(row[\'code\'])\n        except SyntaxError:\n            syntax_error_rows.append(index)\n    torch.cuda.synchronize()\n    phases[\'write_and_validate\'] = time.perf_counter() - phase_started\n    seconds = time.perf_counter() - begin\n    result = {\'gpu\': gpu, \'parameters\': parameters, \'rows\': len(checked), \'cold_seconds\': seconds,\n              \'max_new_tokens\': settings[\'max_new_tokens\'], \'batch_size\': batch_size, \'no_eos\': no_eos,\n              \'source_clipped\': source_clipped, \'checkpoint_sha256\': checkpoint_sha256,\n              \'actual_t4\': actual_t4, \'within_180_seconds\': seconds <= 180,\n              \'gpu_capability\': list(capability), \'gpu_memory_bytes\': torch.cuda.get_device_properties(0).total_memory,\n              \'peak_gpu_memory_bytes\': torch.cuda.max_memory_allocated(),\n              \'python_version\': platform.python_version(), \'torch_version\': torch.__version__,\n              \'transformers_version\': transformers.__version__, \'cuda_version\': torch.version.cuda,\n              \'phase_seconds\': phases, \'syntax_error_rows\': syntax_error_rows,\n              \'empty_code_rows\': [i for i, row in enumerate(checked, 1) if not row[\'code\'].strip()],\n              \'questions_lf_sha256\': normalized_questions_sha256,\n              \'submission_sha256\': hashlib.sha256(output.read_bytes()).hexdigest(),\n              \'questions_sha256\': hashlib.sha256(Path(args.questions).read_bytes()).hexdigest()}\n    output.with_suffix(\'.timing.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    print(json.dumps(result, indent=2))\n    if not args.allow_other_gpu and seconds > 180:\n        raise RuntimeError(\'T4 180-second requirement failed\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
Path('inference_130m.py').write_text(INFERENCE_PROGRAM, encoding="utf-8")
BENCHMARK_PROGRAM = '"""Time a new inference process; retain diagnostics even on failure or timeout."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport time\n\n\ndef run_benchmark(command, output, *, timeout=190, allow_other_gpu=False):\n    output = Path(output)\n    timing = output.with_suffix(\'.timing.json\')\n    log = output.with_suffix(\'.log\')\n    # A failed rerun must never expose a previous submission as its output.\n    timing.unlink(missing_ok=True)\n    output.unlink(missing_ok=True)\n    started = time.perf_counter()\n    returncode = None\n    error = None\n    timed_out = False\n    with log.open(\'w\', encoding=\'utf-8\') as stream:\n        try:\n            process = subprocess.run(command, timeout=timeout, stdout=stream, stderr=subprocess.STDOUT)\n            returncode = process.returncode\n        except subprocess.TimeoutExpired:\n            timed_out = True\n            error = f\'Inference exceeded the {timeout:g}-second process timeout\'\n        except OSError as exc:\n            error = f\'{type(exc).__name__}: {exc}\'\n    seconds = time.perf_counter() - started\n    result = {}\n    if timing.exists():\n        try:\n            result = json.loads(timing.read_text(encoding=\'utf-8\'))\n            if not isinstance(result, dict):\n                raise ValueError(\'Expected a JSON object\')\n        except (ValueError, OSError) as exc:\n            result = {}\n            error = f\'Invalid inference report: {exc}\'\n    complete = (returncode == 0 and not timed_out and error is None\n                and output.is_file() and result.get(\'rows\') == 1919\n                and result.get(\'parameters\') == 130_183_936)\n    verified = complete and result.get(\'actual_t4\') is True and seconds <= 180\n    success = complete and (verified or allow_other_gpu)\n    if not success and error is None:\n        error = (\'The real T4 run exceeded 180 seconds or used a different GPU\'\n                 if complete else f\'Inference failed or did not write complete output (exit {returncode})\')\n    result.update(status=\'completed\' if success else \'failed\',\n                  cold_subprocess_seconds=seconds, process_returncode=returncode,\n                  timed_out=timed_out, process_timeout_seconds=timeout,\n                  t4_limit_verified=verified, within_180_seconds=complete and seconds <= 180,\n                  log_file=str(log), error=error)\n    timing.write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    print(json.dumps(result, indent=2), flush=True)\n    if not success:\n        raise RuntimeError(f\'{error}. See {log} and {timing}\')\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--checkpoint\', default=\'checkpoint_130m.zip\')\n    parser.add_argument(\'--questions\', default=\'questions_used.jsonl\')\n    parser.add_argument(\'--output\', default=\'submission_130m.jsonl\')\n    parser.add_argument(\'--batch-size\', type=int, default=128)\n    parser.add_argument(\'--max-new-tokens\', type=int, default=192)\n    parser.add_argument(\'--timeout\', type=float, default=190)\n    parser.add_argument(\'--allow-other-gpu\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.timeout <= 0:\n        parser.error(\'--timeout must be positive\')\n    command = [sys.executable, \'-u\', str(Path(__file__).with_name(\'inference_130m.py\')),\n               \'--checkpoint\', args.checkpoint, \'--questions\', args.questions,\n               \'--output\', args.output, \'--batch-size\', str(args.batch_size),\n               \'--max-new-tokens\', str(args.max_new_tokens)]\n    if args.allow_other_gpu:\n        command.append(\'--allow-other-gpu\')\n    run_benchmark(command, args.output, timeout=args.timeout, allow_other_gpu=args.allow_other_gpu)\n\n\nif __name__ == \'__main__\':\n    main()\n'
Path('benchmark_130m.py').write_text(BENCHMARK_PROGRAM, encoding="utf-8")


Таймер запускается до нового Python-процесса. Время включает импорты, проверку и распаковку архива, загрузку модели, токенизацию, генерацию 1919 программ и запись JSONL. Установка пакетов и загрузка входных файлов выполняются заранее. Проверка требует actual_t4=true и не больше 180 секунд.

In [ ]:
import subprocess, sys, json
BATCH_SIZE = 128
with zipfile.ZipFile('checkpoint_130m.zip') as archive:
    settings = json.loads(archive.read('best/inference_config.json'))
MAX_NEW_TOKENS = settings['max_new_tokens']
print('Лимит токенов:', MAX_NEW_TOKENS)
subprocess.run([sys.executable, '-u', 'benchmark_130m.py', '--questions', QUESTIONS_FILE,
                '--batch-size', str(BATCH_SIZE), '--max-new-tokens', str(MAX_NEW_TOKENS)],
               check=True, timeout=230)
report = json.loads(Path('submission_130m.timing.json').read_text(encoding='utf-8'))
assert report['rows'] == 1919 and report['parameters'] == 130183936
assert report['actual_t4'] and report['t4_limit_verified'] and report['status'] == 'completed'
print('Все 1919 строк на одной T4:', report['cold_subprocess_seconds'], 'секунд')
print('Незавершённых генераций:', report['no_eos'], 'синтаксические ошибки в строках:', report['syntax_error_rows'])


В конце должны быть rows=1919, actual_t4=true, t4_limit_verified=true и cold_subprocess_seconds <= 180. Скачайте оба файла по ссылкам ниже и сохраните выполненный ноутбук через Save Version → Save & Run All. При нехватке памяти измените BATCH_SIZE на 64 и повторите запуск.


In [ ]:
from IPython.display import FileLink, display
for name in ['submission_130m.jsonl', 'submission_130m.timing.json', 'submission_130m.log', 'hardware_preflight.txt']:
    display(FileLink(name))
